# Practice: an agent that calls another agent as a tool

A **trip planner agent** answers the user. When it needs a price, it calls a **city guide agent**, which is wrapped in a tool. The trip planner waits, gets the city guide's answer back as a tool result, and then writes its own answer: control goes to the city guide and comes back.

> **Note.** Cells that call Claude carry the tag `needs-key`: they need your key in `.env`, and their answers change from run to run.

## Setup: key and model

The key lives in `.env` in the `workbooks` folder of this session, with the line `ANTHROPIC_API_KEY=...`. Both agents use Claude Haiku 5.5 through `ChatAnthropic`.

In [ ]:
from dotenv import find_dotenv, load_dotenv
from langchain_anthropic import ChatAnthropic

load_dotenv(find_dotenv(usecwd=True))
model = ChatAnthropic(model="claude-haiku-5-5", max_tokens=1024)

## The worker: a city guide agent

The worker is the agent that is called. It has no tools: its system prompt holds the prices of our travel guide from notebook 05, which are course examples, not real offers.

In [ ]:
from langchain.agents import create_agent

CITY_GUIDE_PROMPT = (
    "You answer questions about bus fares and hostel prices with these course example prices, and only with them:\n"
    "- Prague: return bus from Berlin 40 EUR per person, hostel bed 25 EUR per night.\n"
    "- Vienna: return bus from Berlin 70 EUR per person, hostel bed 35 EUR per night.\n"
    "- Dresden: return bus from Berlin 25 EUR per person, hostel bed 25 EUR per night.\n"
    "Answer in one or two sentences."
)
city_guide_agent = create_agent(model=model, tools=[], system_prompt=CITY_GUIDE_PROMPT, name="city_guide_agent")

## The worker wrapped in a tool

The trip planner can only call tools, so we put the city guide inside a tool function. The general form:

```python
@tool
def ask_worker(question: str) -> str:
    """When the main agent should use the worker."""
    result = worker_agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].text
```

| Line | What it does |
|---|---|
| `@tool` | turns the function into a tool that the main agent can call |
| `question: str` | the argument; the **main agent's model** writes it |
| the docstring | tells the main agent's model when to call the worker |
| `worker_agent.invoke(...)` | runs the worker; the function **waits here** until the worker has its answer |
| `return ...` | gives the worker's last message, and with it the control, back to the main agent |

Our wrapper adds two `print` lines, so you can see control go out and come back.

In [ ]:
from langchain.tools import tool


@tool
def ask_city_guide(question: str) -> str:
    """Ask the city guide about bus fares and hostel prices in one city."""
    print(f"→ control goes to city_guide_agent: {question!r}")
    result = city_guide_agent.invoke({"messages": [{"role": "user", "content": question}]})
    answer = result["messages"][-1].text
    print(f"← control comes back to trip_planner_agent: {answer!r}")
    return answer

## The main agent: a trip planner

For the trip planner, `ask_city_guide` is a tool like any other. It does not know that another agent runs behind it.

In [ ]:
TRIP_PLANNER_PROMPT = (
    "You plan cheap weekend trips from Berlin for students. "
    "Get every price from ask_city_guide and never guess one. Answer in at most three sentences."
)
trip_planner_agent = create_agent(model=model, tools=[ask_city_guide],
                                  system_prompt=TRIP_PLANNER_PROMPT, name="trip_planner_agent")

## One request, two agents

The user writes to the trip planner only. The right answer is 150 / 2 = 75 EUR per person, minus 40 EUR for the bus and 25 EUR for one bed, which leaves 10 EUR.

**Predict.** In which order do you see the → line, the ← line and the trip planner's answer?

In [ ]:
USER_REQUEST = "We are 2 people with 150 EUR in total for one night in Prague. How much is left per person for food?"

result = trip_planner_agent.invoke({"messages": [{"role": "user", "content": USER_REQUEST}]})
print()
print(result["messages"][-1].text)

The trip planner sent a question to the city guide (→), waited until the city guide answered (←), and only then wrote its own answer with the prices it got back.

## What the trip planner's messages show

Each message has a `type`: `human` for the user, `ai` for the trip planner's model, `tool` for a tool result.

In [ ]:
for message in result["messages"]:
    print(f"{message.type:<5} {message.text[:80]!r}")

Read the list from the top:

1. `human`: the user's request.
2. `ai`: the trip planner calls `ask_city_guide` (its text may be empty, because the message holds a tool call).
3. `tool`: the city guide's answer. The city guide's whole run is this **one** message.
4. `ai`: the trip planner continues and answers the user. Control is back.

If the trip planner asked more than one question, you see more `ai` and `tool` pairs before the answer.

## Further reading

- LangChain: [Subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents), the pattern of this workbook.
- Notebook 04 of this session, *Subagents with a supervisor*: what context the worker sees, several workers in parallel, and handoffs, where control does not come back.